# Crop Disease Detection using CNN vs HOG + HSV + SVM

**Objective:** Load the crop-leaf image dataset from `crop_leaf_dataset.csv`, train two classifiers on the same data, test both on the held-out test set, evaluate them using multiple metrics, visualize the results, and automatically select the algorithm with the higher test accuracy.

**Algorithms**
1. HOG + HSV + RBF-SVM
2. Convolutional Neural Network (CNN)

> **Folder requirement:** Keep this notebook, `crop_leaf_dataset.csv`, and the `images/` folder in the same project folder. The CSV must contain an `image_path` column pointing to the actual image files.


In [1]:
# If required, uncomment and run this installation cell once.
# %pip install numpy pandas matplotlib seaborn opencv-python scikit-image scikit-learn tensorflow pillow


In [2]:
# 1. Import libraries
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from skimage.feature import hog

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

np.random.seed(42)
tf.random.set_seed(42)

print("Libraries loaded successfully.")
print("TensorFlow version:", tf.__version__)


ModuleNotFoundError: No module named 'skimage'

In [ ]:
# 2. Load CSV
CSV_FILE = "crop_leaf_dataset.csv"

if not Path(CSV_FILE).exists():
    raise FileNotFoundError(
        f"{CSV_FILE} was not found. Put the CSV in the same folder as this notebook."
    )

df = pd.read_csv(CSV_FILE)

print("Dataset loaded successfully")
print("Number of rows:", len(df))
print("\nColumns:")
print(df.columns.tolist())
display(df.head())


In [ ]:
# 3. Make the CSV robust to common column names
# The preferred columns are: image_path, label, class_name, split.

if "image_path" not in df.columns:
    # Try common alternatives.
    alternatives = ["path", "filepath", "file_path", "image", "filename"]
    found = next((c for c in alternatives if c in df.columns), None)
    if found:
        df["image_path"] = df[found]
    else:
        raise ValueError(
            "The CSV does not contain an image_path column (or a recognized alternative). "
            "The CSV needs paths to the actual images."
        )

if "label" not in df.columns:
    if "class_name" in df.columns:
        classes = {name: i for i, name in enumerate(sorted(df["class_name"].unique()))}
        df["label"] = df["class_name"].map(classes)
    else:
        raise ValueError("CSV needs either 'label' or 'class_name'.")

if "class_name" not in df.columns:
    df["class_name"] = df["label"].map({0: "Healthy", 1: "Diseased"})

# Convert relative paths consistently.
df["image_path"] = df["image_path"].astype(str).apply(
    lambda p: str(Path(p).as_posix())
)

print("Final columns:", df.columns.tolist())
print("\nClass distribution:")
display(df["class_name"].value_counts().to_frame("count"))


In [ ]:
# 4. Check whether every image exists
missing = []

for p in df["image_path"]:
    if not Path(p).exists():
        missing.append(p)

print("Total images :", len(df))
print("Missing files:", len(missing))

if missing:
    print("\nFirst missing paths:")
    for p in missing[:20]:
        print(p)
    raise FileNotFoundError(
        "\nImages are missing. Fix the image_path values or place the images "
        "in the expected folders before continuing."
    )

print("All image files found.")


In [ ]:
# 5. Dataset distribution bar graph
counts = df["class_name"].value_counts()

plt.figure(figsize=(8, 5))
counts.plot(kind="bar")
plt.title("Crop Leaf Dataset Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()


In [ ]:
# 6. Display sample images
sample_df = df.sample(min(10, len(df)), random_state=42)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    img = cv2.imread(row["image_path"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(row["class_name"])
    ax.axis("off")

for ax in axes[len(sample_df):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 7. Pixel-intensity histogram
plt.figure(figsize=(10, 5))

for class_name in df["class_name"].unique():
    subset = df[df["class_name"] == class_name].sample(
        min(10, len(df[df["class_name"] == class_name])),
        random_state=42
    )
    pixels = []

    for _, row in subset.iterrows():
        img = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE)
        pixels.extend(img.ravel())

    plt.hist(pixels, bins=50, alpha=0.5, label=class_name)

plt.title("Pixel Intensity Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")
plt.legend()
plt.show()


In [ ]:
# 8. Create train/test dataframes
if "split" in df.columns:
    train_df = df[df["split"].str.lower() == "train"].copy()
    test_df  = df[df["split"].str.lower() == "test"].copy()
else:
    # If no split column exists, create a reproducible stratified 80/20 split.
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(
        df,
        test_size=0.20,
        random_state=42,
        stratify=df["label"]
    )

print("Training samples:", len(train_df))
print("Testing samples :", len(test_df))
print("\nTraining distribution:")
display(train_df["class_name"].value_counts().to_frame("count"))
print("\nTesting distribution:")
display(test_df["class_name"].value_counts().to_frame("count"))


## Algorithm 1 — HOG + HSV + SVM

HOG captures local edge/orientation information, while HSV statistics provide colour information. The combined feature vector is standardised and classified using an RBF-kernel SVM.


In [ ]:
# 9. HOG + HSV feature extraction
IMG_SIZE = (128, 128)

def extract_hog_hsv(image_path):
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Could not read image: {image_path}")

    image = cv2.resize(image, IMG_SIZE)

    # HOG features
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    # HSV colour statistics
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hsv_features = []

    for channel in cv2.split(hsv):
        hsv_features.extend([
            np.mean(channel),
            np.std(channel),
            np.min(channel),
            np.max(channel)
        ])

    return np.concatenate([
        hog_features,
        np.asarray(hsv_features, dtype=np.float32)
    ])

X_train_svm = np.array([extract_hog_hsv(p) for p in train_df["image_path"]])
X_test_svm  = np.array([extract_hog_hsv(p) for p in test_df["image_path"]])

y_train = train_df["label"].astype(int).values
y_test  = test_df["label"].astype(int).values

print("SVM training feature shape:", X_train_svm.shape)
print("SVM testing feature shape :", X_test_svm.shape)


In [ ]:
# 10. Train HOG + HSV + SVM
svm_model = make_pipeline(
    StandardScaler(),
    SVC(
        kernel="rbf",
        C=5,
        gamma="scale",
        probability=True,
        random_state=42
    )
)

svm_model.fit(X_train_svm, y_train)

svm_pred = svm_model.predict(X_test_svm)
svm_prob = svm_model.predict_proba(X_test_svm)[:, 1]

print("SVM training completed.")


In [ ]:
# 11. Evaluate SVM
svm_accuracy  = accuracy_score(y_test, svm_pred)
svm_precision = precision_score(y_test, svm_pred, zero_division=0)
svm_recall    = recall_score(y_test, svm_pred, zero_division=0)
svm_f1        = f1_score(y_test, svm_pred, zero_division=0)
svm_auc       = roc_auc_score(y_test, svm_prob)

print("===== HOG + HSV + SVM RESULTS =====")
print(f"Accuracy  : {svm_accuracy*100:.2f}%")
print(f"Precision : {svm_precision*100:.2f}%")
print(f"Recall    : {svm_recall*100:.2f}%")
print(f"F1 Score  : {svm_f1*100:.2f}%")
print(f"ROC-AUC   : {svm_auc:.3f}")

print("\nClassification Report:")
print(classification_report(
    y_test, svm_pred,
    target_names=["Healthy", "Diseased"],
    zero_division=0
))


## Algorithm 2 — Convolutional Neural Network

The CNN learns image features directly from RGB pixels. Data augmentation is applied only to the training pipeline. The test images remain untouched for final evaluation.


In [ ]:
# 12. Load RGB images for CNN
def load_images(dataframe):
    X, y = [], []

    for _, row in dataframe.iterrows():
        img = cv2.imread(str(row["image_path"]))

        if img is None:
            raise ValueError(f"Could not read image: {row['image_path']}")

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)

        X.append(img.astype(np.float32) / 255.0)
        y.append(int(row["label"]))

    return np.array(X), np.array(y)

X_train_cnn, y_train_cnn = load_images(train_df)
X_test_cnn, y_test_cnn = load_images(test_df)

print("CNN training shape:", X_train_cnn.shape)
print("CNN testing shape :", X_test_cnn.shape)


In [ ]:
# 13. Build CNN
cnn_model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),

    layers.Dense(1, activation="sigmoid")
])

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()


In [ ]:
# 14. Train CNN
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = cnn_model.fit(
    X_train_cnn,
    y_train_cnn,
    validation_split=0.15,
    epochs=25,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

print("CNN training completed.")


In [ ]:
# 15. CNN training/validation accuracy graph
plt.figure(figsize=(10, 5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("CNN Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# 16. CNN training/validation loss graph
plt.figure(figsize=(10, 5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("CNN Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# 17. Test CNN
cnn_prob = cnn_model.predict(X_test_cnn, verbose=0).ravel()
cnn_pred = (cnn_prob >= 0.5).astype(int)

cnn_accuracy  = accuracy_score(y_test_cnn, cnn_pred)
cnn_precision = precision_score(y_test_cnn, cnn_pred, zero_division=0)
cnn_recall    = recall_score(y_test_cnn, cnn_pred, zero_division=0)
cnn_f1        = f1_score(y_test_cnn, cnn_pred, zero_division=0)
cnn_auc       = roc_auc_score(y_test_cnn, cnn_prob)

print("===== CNN RESULTS =====")
print(f"Accuracy  : {cnn_accuracy*100:.2f}%")
print(f"Precision : {cnn_precision*100:.2f}%")
print(f"Recall    : {cnn_recall*100:.2f}%")
print(f"F1 Score  : {cnn_f1*100:.2f}%")
print(f"ROC-AUC   : {cnn_auc:.3f}")

print("\nClassification Report:")
print(classification_report(
    y_test_cnn, cnn_pred,
    target_names=["Healthy", "Diseased"],
    zero_division=0
))


In [ ]:
# 18. Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_svm = confusion_matrix(y_test, svm_pred)
cm_cnn = confusion_matrix(y_test_cnn, cnn_pred)

sns.heatmap(
    cm_svm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Healthy", "Diseased"],
    yticklabels=["Healthy", "Diseased"],
    ax=axes[0]
)
axes[0].set_title("HOG + HSV + SVM")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(
    cm_cnn, annot=True, fmt="d", cmap="Greens",
    xticklabels=["Healthy", "Diseased"],
    yticklabels=["Healthy", "Diseased"],
    ax=axes[1]
)
axes[1].set_title("CNN")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.show()


In [ ]:
# 19. ROC curves
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_prob)
fpr_cnn, tpr_cnn, _ = roc_curve(y_test_cnn, cnn_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr_svm, tpr_svm, label=f"SVM (AUC = {svm_auc:.3f})")
plt.plot(fpr_cnn, tpr_cnn, label=f"CNN (AUC = {cnn_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# 20. Final comparison table
results = pd.DataFrame({
    "Algorithm": ["HOG + HSV + SVM", "CNN"],
    "Accuracy (%)": [svm_accuracy*100, cnn_accuracy*100],
    "Precision (%)": [svm_precision*100, cnn_precision*100],
    "Recall (%)": [svm_recall*100, cnn_recall*100],
    "F1 Score (%)": [svm_f1*100, cnn_f1*100],
    "ROC-AUC": [svm_auc, cnn_auc]
})

display(results.round(2))


In [ ]:
# 21. Comparative metric bar graph
results_plot = results.set_index("Algorithm")

ax = results_plot[[
    "Accuracy (%)",
    "Precision (%)",
    "Recall (%)",
    "F1 Score (%)"
]].plot(kind="bar", figsize=(11, 6))

plt.title("CNN vs HOG + HSV + SVM")
plt.ylabel("Score (%)")
plt.xlabel("Algorithm")
plt.ylim(0, 105)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 22. Sample predictions
label_names = {0: "Healthy", 1: "Diseased"}

sample_count = min(8, len(test_df))
sample_indices = np.linspace(0, len(test_df)-1, sample_count, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.ravel()

for ax, idx in zip(axes, sample_indices):
    row = test_df.iloc[idx]
    img = cv2.imread(row["image_path"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    svm_label = label_names[int(svm_pred[idx])]
    cnn_label = label_names[int(cnn_pred[idx])]

    ax.imshow(img)
    ax.set_title(
        f"Actual: {label_names[int(row['label'])]}\n"
        f"SVM: {svm_label} | CNN: {cnn_label}"
    )
    ax.axis("off")

for ax in axes[sample_count:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# 23. Automatic final model selection
if cnn_accuracy > svm_accuracy:
    final_algorithm = "CNN"
    final_accuracy = cnn_accuracy
elif svm_accuracy > cnn_accuracy:
    final_algorithm = "HOG + HSV + SVM"
    final_accuracy = svm_accuracy
else:
    final_algorithm = "Tie"
    final_accuracy = cnn_accuracy

print("=" * 65)
print("FINAL ALGORITHM SELECTION")
print("=" * 65)
print(f"HOG + HSV + SVM Accuracy : {svm_accuracy*100:.2f}%")
print(f"CNN Accuracy             : {cnn_accuracy*100:.2f}%")
print("-" * 65)
print("Selected Algorithm       :", final_algorithm)
print(f"Final Test Accuracy      : {final_accuracy*100:.2f}%")
print("=" * 65)


In [ ]:
# 24. Save results to CSV
results.to_csv("model_comparison_results.csv", index=False)

prediction_output = test_df.copy()
prediction_output["svm_prediction"] = svm_pred
prediction_output["svm_probability_diseased"] = svm_prob
prediction_output["cnn_prediction"] = cnn_pred
prediction_output["cnn_probability_diseased"] = cnn_prob

prediction_output.to_csv(
    "test_predictions.csv",
    index=False
)

print("Saved:")
print(" - model_comparison_results.csv")
print(" - test_predictions.csv")


## Final interpretation

The final model is selected **from the measured test accuracy**, not from a hard-coded value. The same held-out test images are used for both algorithms.

For the project report, record:
- Dataset size and class distribution
- HOG + HSV + SVM test metrics
- CNN test metrics
- Confusion matrices
- ROC curves
- Training/validation graphs
- Comparative bar graph
- Final selected algorithm and test accuracy

**Important:** Do not manually edit the accuracy values. If CNN does not outperform SVM on your particular image set, improve the training configuration or dataset quality and rerun the experiment rather than changing the printed results.
